# Principal Component Analysis (PCA)

Principal Component Analysis (PCA) summarizes genome-wide genotype variation into a small number of axes called principal components (PCs). In population genetics, the first few PCs often capture broad ancestry structure, sample outliers, or batch effects.

This tutorial shows how to:

- load a representative subset of 1000 Genomes Project samples with population metadata,
- perform PCA with the scikit-learn CPU backend using low-rank SVD,
- perform PCA with the PyTorch backend using exact SVD on a GPU when available,
- calculate PC coordinates in one step or compute eigenvectors first and then project samples,
- visualize PC coordinates and color samples by population metadata.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from snputils.datasets import load_dataset
from snputils.processing import PCA, embedding_dataframe_from_model
from snputils.visualization import plot_embedding

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "0.9",
    "grid.linewidth": 0.8,
})


## Load a 1000 Genomes Subset

The code below loads 12 populations from the 1000 Genomes Project Phase 3 panel, taking up to 50 samples per population and up to 100,000 SNPs.

The selected population labels are stored on the returned `SNPObject` as `sample_fid`, aligned to `snpobj.samples`. The genotype array is kept as two strands per individual (`sum_strands=False`); the PCA examples below average those strands into one genotype dosage per sample.

In [ ]:
DATA_DIR = Path("data/1kgp_pca")

populations = [
    "YRI", "LWK", "MSL", "GWD",
    "CEU", "FIN", "GBR", "IBS",
    "CHB", "JPT", "GIH", "PEL",
]

snpobj = load_dataset(
    "1kgp",
    resource="phase3",
    output_dir=DATA_DIR,
    populations=populations,
    samples_per_population=50,
    max_variants=100_000,
    require_biallelic=True,
    require_complete=True,
    require_polymorphic=True,
    snv_only=True,
    sum_strands=False,
)

print(f"Loaded {snpobj.n_samples:,} samples and {snpobj.n_snps:,} variants")
print(f"Genotype array shape: {snpobj.genotypes.shape}")

Create a metadata table that will be reused when plotting PC coordinates. Each row represents one individual.

In [ ]:
sample_metadata = pd.DataFrame({
    "sample": snpobj.samples,
    "population": snpobj.sample_fid,
    "sex": snpobj.sample_sex,
})

sample_metadata.head()

## Build Tables and Plots with snputils Utilities

`PCA.fit_transform()` and `PCA.transform()` store projected coordinates on the fitted object as `X_new_`. snputils provides `embedding_dataframe_from_model()` to convert those coordinates into a table with `PC1`, `PC2`, and sample identifiers, and `plot_embedding()` to make metadata-colored scatter plots from that table.


## PCA on CPU with Low-Rank SVD

The scikit-learn backend runs on CPU. Setting `fitting="lowrank"` uses a randomized low-rank SVD, which is often a practical choice when the goal is to calculate the first few PCs from a large genotype matrix.

Here `fit_transform()` computes the eigenvectors and immediately projects the same samples onto the requested PCs.

In [ ]:
pca_cpu = PCA(
    backend="sklearn",
    n_components=10,
    fitting="lowrank",
    average_strands=True,
)

coords_cpu = pca_cpu.fit_transform(snpobj)
coords_cpu_df = embedding_dataframe_from_model(pca_cpu, metadata=sample_metadata)
coords_cpu_df.head()

In [ ]:
plot_embedding(coords_cpu_df, title="1000 Genomes PCA: CPU low-rank SVD", hue="population")
plt.show()

## Compute Eigenvectors, Then Project Samples

Sometimes it is useful to separate the two steps:

1. `fit()` computes the PCA eigenvectors from a set of samples and variants.
2. `transform()` projects genotype data onto those PCs.

The example below computes eigenvectors from the loaded dataset and then projects the same samples. The projected coordinates are equivalent, up to tiny numerical differences and the arbitrary sign of each PC.

In [ ]:
pca_two_step = PCA(
    backend="sklearn",
    n_components=4,
    fitting="lowrank",
    average_strands=True,
)

pca_two_step.fit(snpobj)
coords_projected = pca_two_step.transform(snpobj)
coords_projected_df = embedding_dataframe_from_model(pca_two_step, metadata=sample_metadata)

coords_projected_df.head()

After `fit()`, the principal axes are available in `components_`. Each row is one eigenvector, and each column corresponds to one variant used in the PCA calculation.

In [ ]:
print(f"Number of PCs computed: {pca_two_step.n_components_}")
print(f"Eigenvector matrix shape: {pca_two_step.components_.shape}")
print(f"Per-variant mean vector shape: {pca_two_step.mean_.shape}")

## PCA on GPU with Exact SVD

The PyTorch backend can run PCA on a CUDA GPU. Setting `fitting="exact"` uses an exact SVD. If CUDA is unavailable, `device="cuda:0"` falls back to CPU with a warning, so the code remains runnable on a laptop while still showing the GPU configuration.

Exact SVD can be useful when you want deterministic full-SVD behavior for a selected number of PCs and have enough GPU memory for the selected genotype matrix.

In [ ]:
pca_gpu = PCA(
    backend="pytorch",
    n_components=10,
    fitting="exact",
    device="cuda:0",
    average_strands=True,
)

coords_gpu = pca_gpu.fit_transform(snpobj)
coords_gpu_df = embedding_dataframe_from_model(pca_gpu, metadata=sample_metadata)

print(f"PyTorch PCA device: {pca_gpu.device}")
coords_gpu_df.head()

In [ ]:
plot_embedding(coords_gpu_df, title="1000 Genomes PCA: PyTorch exact SVD", hue="population")
plt.show()

## Color PCA Plots by Population Groups

The population metadata can also be collapsed into broader groups for a less crowded plot. The grouping below is only for visualization; the PCA coordinates are the same.

In [ ]:
population_group = {
    "YRI": "Africa", "LWK": "Africa", "MSL": "Africa", "GWD": "Africa",
    "CEU": "Europe", "FIN": "Europe", "GBR": "Europe", "IBS": "Europe",
    "CHB": "East Asia", "JPT": "East Asia",
    "GIH": "South Asia",
    "PEL": "Americas",
}

coords_cpu_df["population_group"] = coords_cpu_df["population"].map(population_group)
plot_embedding(coords_cpu_df, title="1000 Genomes PCA colored by population group", hue="population_group")
plt.show()

## Using Sample and Variant Subsets

`PCA` can select rows and columns before computing PCs. `samples_subset` chooses samples by row index, and `snps_subset` chooses variants by column index. Passing an integer uses the first `n` rows or columns; passing a list uses the specified indices.

This is helpful for quick exploratory PCA calculations before scaling up to the full selected dataset.

In [ ]:
pca_subset = PCA(
    backend="sklearn",
    n_components=4,
    fitting="lowrank",
    average_strands=True,
    samples_subset=200,
    snps_subset=20_000,
)

coords_subset = pca_subset.fit_transform(snpobj)
coords_subset_df = embedding_dataframe_from_model(pca_subset, metadata=sample_metadata)

print(f"Subset PC coordinate shape: {coords_subset.shape}")
plot_embedding(coords_subset_df, title="PCA on the first 200 samples and 20,000 variants", hue="population")
plt.show()

## Treating Strands as Separate Rows

By default, `average_strands=True` averages the two genotype strands into one dosage row per individual. If you set `average_strands=False`, PCA treats each strand as a separate row. In that case, the output has two rows per diploid sample, and `pca.haplotypes_` identifies the strand-specific rows.

In [ ]:
pca_haplotypes = PCA(
    backend="sklearn",
    n_components=2,
    fitting="lowrank",
    average_strands=False,
    snps_subset=10_000,
)

coords_hap = pca_haplotypes.fit_transform(snpobj)
coords_hap_df = embedding_dataframe_from_model(pca_haplotypes, metadata=sample_metadata)

coords_hap_df.head()


In [ ]:
plot_embedding(coords_hap_df, title="PCA with each strand represented separately", hue="population")
plt.show()

## Save PC Coordinates

If `embedding_table_path` is set, `fit_transform()` writes the projected PC coordinates to a TSV or CSV file. This is useful when PC coordinates will be used by another analysis or plotting script.

In [ ]:
output_path = DATA_DIR / "pca_coordinates.tsv"

pca_export = PCA(
    backend="sklearn",
    n_components=4,
    fitting="lowrank",
    average_strands=True,
    embedding_table_path=output_path,
)
_ = pca_export.fit_transform(snpobj)

print(f"Wrote PCA coordinates to {output_path}")